# 03 Universal Reconstruction 50MP (Phase 4)

**Synthetic RAW 50MP Lab.** This notebook processes either kind of source image, a face or crowd scene or a
night scene, with one shared pipeline and adaptive settings, and ends with **one final full-size 50 MP image**
(PRD sections 4.3, 20 and 30 Phase 4):

- **Scene analysis and classification:** scene statistics (tone distribution, dark share, luminance regions,
  colourfulness) and optional face detection classify the scene as `human`, `night`, `night_with_faces` or
  `general`, with the cues, thresholds and reasons recorded. Low-confidence decisions use conservative general
  settings and ask you to confirm (PRD 20.2).
- **Automatic preset selection:** human scenes go to the notebook 01 human and crowd pipeline, night scenes
  to the notebook 02 night pipeline, and everything else (including night scenes with faces) to the universal
  common pipeline: Lanczos, the SR ladder (Phase 2 backend), combined region and face masks, universal tone,
  colour and region-aware detail, face protection.
- **Default policy:** the most conservative settings that produce measurable improvement. The SR ladder holds
  only the conservative mode by default, and **no SR result that failed a seam gate reaches the final image**.
- **Standardised evaluation:** every route's final master is measured against the Lanczos baseline in the same
  way (detail energy, source consistency, clipping).
- **Final image:** like every reconstruction notebook, the run ends with its one final 50 MP image: a 16-bit
  TIFF and an EXR saved under `outputs/universal/<run id>/final/` by the shared final-output step, verified by
  SHA-256, downloaded in Colab and copied to Drive when it is mounted (section 18).

All processing code lives in the shared `synthetic_raw_lab` package (`src/scene_analysis.py`,
`src/universal_config.py`, `src/universal_pipeline.py`, reusing the human, night and Phase 2 modules); this
notebook only sets parameters, calls the package and displays results.

## 1. Introduction and scientific limitations

- The source is a rendered, **display-referred** image. Its linear-light values are not scene-referred radiance,
  and nothing here recovers sensor measurements or clipped highlights.
- Pretrained super-resolution **synthesises** detail above the source sampling limit from learned priors. That
  detail is inferred, not recovered, and may be wrong even when it looks plausible.
- **Classification is a processing heuristic.** Its thresholds are provisional, to be calibrated
  experimentally; zero detected faces does not mean there are no faces, and a detected face is never identity
  proof. Check section 8 and override the category when it is wrong.
- **Measurable improvement** is measured with provisional gates: more high-frequency energy than Lanczos without
  losing source consistency or adding clipping. High-frequency energy also rises with ringing, noise and seams,
  so the crops at 100% (section 16) remain the acceptance check.
- Outputs are **synthetic camera-grade masters**, never camera RAW files.

## 2. Runtime setup

Set the run parameters. In Google Colab these are form fields. Use a **GPU runtime** (Runtime, Change runtime
type): 50 MP super-resolution takes minutes on a GPU and much longer on the CPU.

- **Repository and code version:** the repository is public, so the setup cell clones `REPO_URL` at `REPO_REF`
  into `/content/experiment_50MP` with no token; `PROJECT_DIR` uses an existing checkout instead.
- **Keep masters:** a 50 MP run writes several hundred MB. Set `MOUNT_DRIVE` and point `OUTPUT_ROOT` (and
  optionally `REPORTS_ROOT`) at Drive so results survive a Colab VM reset.
- **Input:** `upload` asks for a file, `path` reads `INPUT_PATH`; `synthetic_crowd`, `synthetic_night` and
  `synthetic_general` generate deterministic test scenes at `SYNTHETIC_SIZE` (the crowd scene supplies its own
  face boxes, since drawn faces are not real faces for a detector).
- **Presets (PRD 21.6):** `standard` makes the full 50 MP target; `quick` a reduced output for checking the flow
  (2 MP for general and night scenes, 1.5x the source pixel count for human scenes).
- **Scene analysis:** `CATEGORY` overrides the automatic classification (`auto` keeps it). `FACE_DETECTION`
  runs `FACE_DETECTOR` (YuNet downloads a checksum-verified 230 KB model once); `MANUAL_FACE_BOXES` adds source
  pixel boxes `[x, y, width, height]` that always count.
- **Super-resolution:** `SR_MODES` is the ladder, most conservative first; a rung is used only when it passes
  the seam gate and the measurable-improvement gates. `conservative` is the default; `conservative,balanced`
  also tries the balanced mode in the common pipeline. Weights (about 135 MB for SwinIR) download once into
  `WEIGHTS_DIR` (default `<project>/weights`; a Drive folder keeps them across sessions).
- **Final image:** `DOWNLOAD_FINAL` downloads the final 16-bit TIFF through the browser at the end (about
  235 MB for a 50 MP image, so it takes a few minutes). With `MOUNT_DRIVE`, the final TIFF and EXR are also
  copied to `FINAL_DRIVE_DIR`, unless `OUTPUT_ROOT` is on Drive already.
- **Advanced (PRD 22.2):** `ADVANCED` is a dictionary of any further profile keys, flat (for example
  `{"crop_count": 4}`) or by section (for example `{"improvement": {"min_detail_gain": 0.05}}` or
  `{"night": {"overrides": {"texture": {"sharpening_strength": 0.2}}}}`). Every key is listed in section 7.

In [ ]:
# @title Run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/universal_default.yaml"  # @param {type:"string"}
INPUT_MODE = "upload"  # @param ["upload", "path", "synthetic_crowd", "synthetic_night", "synthetic_general"]
INPUT_PATH = ""  # @param {type:"string"}
SYNTHETIC_SIZE = "1024x768"  # @param {type:"string"}
RUNTIME_PRESET = "standard"  # @param ["quick", "standard", "research"]
TARGET_MEGAPIXELS = 50.0  # @param {type:"number"}
CROP_SIZE = 256  # @param {type:"integer"}

# @markdown Scene analysis (PRD 20.2)
CATEGORY = "auto"  # @param ["auto", "human", "night", "night_with_faces", "general"]
FACE_DETECTION = True  # @param {type:"boolean"}
FACE_DETECTOR = "yunet"  # @param ["yunet", "haar"]
MANUAL_FACE_BOXES = []  # @param {type:"raw"}

# @markdown Super-resolution (Phase 2 backend)
ENABLE_SUPER_RESOLUTION = True  # @param {type:"boolean"}
SR_BACKEND = "swinir"  # @param ["swinir", "realesrgan"]
SR_MODES = "conservative"  # @param ["conservative", "conservative,balanced"]
WEIGHTS_DIR = ""  # @param {type:"string"}

# @markdown Final image
DOWNLOAD_FINAL = True  # @param {type:"boolean"}
FINAL_DRIVE_DIR = "/content/drive/MyDrive/experiment_50MP/final"  # @param {type:"string"}

# @markdown Advanced parameter dictionary
ADVANCED = {}  # @param {type:"raw"}

## 3. Dependency installation

Finds the repository (`PROJECT_DIR`, the checkout this notebook lives in, or the Colab clone of `REPO_REF`)
and installs the `synthetic_raw_lab` package from it (plus the `sr` extra, PyTorch and spandrel, when
super-resolution is enabled), like notebooks 00 to 02. The exact package versions used are recorded in every
run manifest.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "03_Universal_50MP/1.0"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git failed (check REPO_URL and REPO_REF): {result.stderr}")


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
missing_sr = ENABLE_SUPER_RESOLUTION and importlib.util.find_spec("spandrel") is None
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src" or missing_sr:
    target = f"{PROJECT}[sr]" if ENABLE_SUPER_RESOLUTION else str(PROJECT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

## 4. GPU and environment inspection

Loads the universal profile (`CONFIG_FILE`, then the form fields and `ADVANCED`; empty path fields keep the
file's paths) so every later cell uses the same settings as the pipeline. Then reports GPU, CUDA driver,
PyTorch (if installed), the device super-resolution will use, RAM and free disk where the masters will be
written (PRD 21.2). Super-resolution uses CUDA when present, then Apple MPS, then the CPU, one network at a
time; interpolation and the tone, colour and detail stages run on the CPU.

In [ ]:
import json

from IPython.display import Image, Markdown, display

from synthetic_raw_lab import config as lab_config
from synthetic_raw_lab import (
    final_output,
    ingestion,
    pipeline,
    resolution,
    sr_models,
    universal_config,
    universal_pipeline,
)
from synthetic_raw_lab import night_processing as night
from synthetic_raw_lab import universal_report as report


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(v).replace("|", "\\|") for v in row) + " |" for row in rows]
    return Markdown("\n".join(lines))


overrides = {
    "runtime_preset": RUNTIME_PRESET,
    "target_megapixels": float(TARGET_MEGAPIXELS),
    "crop_size": CROP_SIZE,
    "analysis": {
        "manual_override": CATEGORY,
        "face_detection": FACE_DETECTION,
        "face_detector": FACE_DETECTOR,
        "manual_face_boxes": [list(box) for box in MANUAL_FACE_BOXES],
    },
    "super_resolution": {
        "enabled": ENABLE_SUPER_RESOLUTION,
        "backend": SR_BACKEND,
        "modes": [m.strip() for m in SR_MODES.split(",") if m.strip()],
    },
}
if WEIGHTS_DIR:
    overrides["super_resolution"]["weights_dir"] = WEIGHTS_DIR
paths = {"output_root": OUTPUT_ROOT, "reports_root": REPORTS_ROOT, "inputs_root": INPUTS_ROOT}
overrides.update({name: value for name, value in paths.items() if value})
for key, value in dict(ADVANCED).items():
    if isinstance(value, dict) and isinstance(overrides.get(key), dict):
        overrides[key] = {**overrides[key], **value}
    else:
        overrides[key] = value
config_path = Path(CONFIG_FILE) if Path(CONFIG_FILE).is_absolute() else PROJECT / CONFIG_FILE
CONFIG = universal_config.load_universal_config(config_path, **overrides)
EXPERIMENT = CONFIG.experiment
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(EXPERIMENT, PROJECT)

runtime = environment.inspect_runtime(OUTPUT_DIR)
gpu = runtime["gpu"]
print(f"Python {runtime['python']} on {runtime['platform']} (Colab: {runtime['in_colab']})")
print(
    f"GPU: {gpu['name']} {gpu['memory_total_gb']} GB, driver CUDA {gpu['driver_cuda_version']}"
    if gpu
    else "GPU: none detected (super-resolution falls back to Apple MPS or the CPU, much slower)"
)
print(f"PyTorch: {runtime['packages'].get('torch', 'not installed')}")
SR = CONFIG.super_resolution
if not SR.enabled:
    print("super-resolution: disabled (every route uses Lanczos interpolation, no SR claimed)")
elif sr_models.available():
    sr_device = sr_models.device_info(sr_models.resolve_device(SR.device))
    print(
        f"super-resolution: {SR.backend} ladder {list(CONFIG.sr_modes)} on {sr_device['name']} ({sr_device['device']})"
    )
else:
    print(f"WARNING: the SR stages will fail: {pipeline.SR_INSTALL_HINT}")
print(
    f"RAM: {runtime['ram_available_gb']} of {runtime['ram_total_gb']} GB available; "
    f"free disk: {runtime['disk_free_gb']} GB; CPUs: {runtime['cpu_count']}"
)
print(f"profile {EXPERIMENT.profile}; final images under {OUTPUT_DIR / EXPERIMENT.input_category}/<run id>/final")

## 5. Input upload

`upload` asks for a file in Colab, `path` uses `INPUT_PATH` (for example a file on Drive), and the
`synthetic_*` modes generate deterministic test scenes at `SYNTHETIC_SIZE`. Supported formats: PNG, JPEG and
TIFF; WebP, AVIF and OpenEXR when the runtime can decode them. The original file is never modified.

In [ ]:
from PIL import Image as PILImage

from synthetic_raw_lab import test_images

# Uploads and generated inputs are staged outside the read-only input store, which ingest() copies into.
STAGING = OUTPUT_DIR / "_staging"
STAGING.mkdir(parents=True, exist_ok=True)

if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("upload only works in Colab; use INPUT_MODE='path' or a synthetic mode")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"upload exactly one image, got {len(uploaded)}")
    name, data = next(iter(uploaded.items()))
    SOURCE = STAGING / name
    SOURCE.write_bytes(data)
elif INPUT_MODE == "path":
    SOURCE = Path(INPUT_PATH).expanduser()
elif INPUT_MODE.startswith("synthetic_"):
    width, height = (int(v) for v in SYNTHETIC_SIZE.lower().split("x"))
    kind = INPUT_MODE.removeprefix("synthetic_")
    SOURCE = STAGING / f"synthetic_{kind}_{width}x{height}.png"
    if kind == "night":
        night.write_night_scene(SOURCE, width, height)
    elif kind == "general":
        test_images.write_test_image(SOURCE, width, height)
    elif kind == "crowd":
        pixels, faces = test_images.make_crowd_test_image(width, height)
        PILImage.fromarray(pixels).save(SOURCE)
        # Drawn faces are not real faces for a detector: supply the ground-truth boxes as manual observations.
        boxes = [[f.x, f.y, f.width, f.height] for f in faces]
        overrides["analysis"] = {**overrides["analysis"], "manual_face_boxes": boxes}
        CONFIG = universal_config.load_universal_config(config_path, **overrides)
        EXPERIMENT = CONFIG.experiment
        print(f"synthetic crowd: {len(boxes)} ground-truth face boxes added as manual observations")
    else:
        raise ValueError(f"unknown INPUT_MODE {INPUT_MODE!r}")
else:
    raise ValueError(f"unknown INPUT_MODE {INPUT_MODE!r}")
print(f"source: {SOURCE} ({SOURCE.stat().st_size:,} bytes)")
display(Image(filename=str(SOURCE), width=720))

## 6. Input validation

Copies the source into the read-only input store, decodes it without channel swapping or tone changes, and
checks dimensions, aspect ratio, bit depth, channels, alpha, ICC profile, EXIF, file size, clipping,
histograms, compression artefacts, corruption and duplicates (PRD 3.4, 7.1).

In [ ]:
source_image = ingestion.ingest(
    SOURCE, INPUTS_DIR, EXPERIMENT.input_category, source_color_space=EXPERIMENT.source_color_space
)
v = source_image.validation
rows = [
    ("format", f"{v['file']['format']}, {v['file']['size_bytes']:,} bytes, sha256 {v['file']['sha256'][:16]}..."),
    ("stored read-only copy", v["file"]["stored_copy"]),
    (
        "dimensions",
        f"{v['dimensions']['width']} x {v['dimensions']['height']} "
        f"({v['dimensions']['megapixels']:.2f} MP, {v['dimensions']['aspect_ratio_reduced']})",
    ),
    ("bit depth", f"{v['samples']['bit_depth']}-bit (effective {v['samples']['effective_bit_depth']}-bit)"),
    ("channels / alpha", f"{v['samples']['channels_in_file']} / {v['alpha']}"),
    ("ICC profile", v["color"]["icc_profile_description"] or "none"),
    ("interpretation", v["color"]["resolved"]["description"] + f" ({v['color']['resolved']['origin']})"),
    ("EXIF", v["metadata"]["exif"] or "none"),
    ("clipping at max", {k: f"{c['at_max_fraction']:.3%}" for k, c in v["clipping"]["channels"].items()}),
    ("8x8 blockiness", f"{v['compression']['blockiness_ratio']:.3f}"),
    ("duplicates", v["duplicates"] or "none"),
]
display(table(["check", "result"], rows))
for warning in source_image.warnings:
    print("WARNING:", warning)

## 7. Experiment configuration

Every parameter of the profile loaded in section 4 (PRD 22.2): the shared experiment settings, the universal
settings (analysis, SR ladder, improvement gates, face protection and the specialised profiles each category is
handed to) and the universal tone, colour and detail settings of the common pipeline. Then the target
resolution of the common pipeline (PRD 3.5), the memory estimate (PRD 21.3) and the storage estimate (PRD 21.5).

In [ ]:
display(
    table(
        ["parameter", "value", "default", "accepted", "effect"],
        [
            (f"`{r['parameter']}`", f"`{r['current']}`", f"`{r['default']}`", r["accepted"], r["effect"])
            for r in lab_config.parameter_table(EXPERIMENT)
        ],
    )
)
display(
    table(
        ["universal parameter", "value", "default"],
        [
            (f"`{r['parameter']}`", f"`{r['current']}`", f"`{r['default']}`")
            for r in universal_config.parameter_rows(CONFIG)
        ],
    )
)
display(
    table(
        ["processing parameter", "value", "accepted", "effect"],
        [
            (f"`{r['parameter']}`", f"`{r['current']}`", r["accepted"], r["effect"])
            for r in night.parameter_table(CONFIG.processing)
            if not r["parameter"].startswith("super_resolution.")
        ],
    )
)

target = resolution.compute_target_resolution(
    source_image.width,
    source_image.height,
    EXPERIMENT.effective_target_megapixels,
    mode=EXPERIMENT.resolution_mode,
    pixel_count_tolerance=EXPERIMENT.pixel_count_tolerance,
    dimension_multiple=EXPERIMENT.dimension_multiple,
    preserve_aspect_ratio=EXPERIMENT.preserve_aspect_ratio,
    max_aspect_ratio_error=EXPERIMENT.max_aspect_ratio_error,
)
print(
    f"target: {target.width} x {target.height} = {target.pixels:,} pixels "
    f"(pixel error {target.pixel_count_error:+.4%}, scale {target.scale_x:.4f} x {target.scale_y:.4f})"
)
for row in environment.estimate_memory(target.width, target.height):
    print(f"  {row['representation']:<28} {row['estimated_mb']:>8.1f} MB")
storage = environment.estimate_storage(
    target.width,
    target.height,
    1,
    tiff=True,
    exr=EXPERIMENT.export_exr,
    exr_bytes_per_sample=2 if EXPERIMENT.exr_precision == "half" else 4,
    source_pixels=source_image.width * source_image.height,
    save_intermediates=EXPERIMENT.save_intermediates,
)
output_free_gb = environment.disk_free_gb(OUTPUT_DIR)
print(
    f"storage upper bound: {storage['upper_bound_bytes'] / 1e9:.2f} GB; free at the output folder: {output_free_gb} GB"
)
if storage["upper_bound_bytes"] > output_free_gb * 1e9:
    print("WARNING: free disk may be insufficient; point OUTPUT_ROOT at Drive or use the quick preset")

## 8. Scene analysis

Measures the scene statistics, observes faces (manual boxes first, then the detector) and classifies the scene
(PRD 20.2): `human` (face- or crowd-dominant, by day or night), `night` (night or low light, no faces),
`night_with_faces` (a night scene with faces that do not dominate it: night policies outside the face masks,
face policies inside) or `general`. The table shows
every cue with its value, threshold and margin. A decision closer to its threshold than
`low_confidence_margin` is low confidence: the conservative general settings are used and you are asked to
confirm. **If the category is wrong, set `CATEGORY` in section 2 and re-run from section 4**; the run in
section 9 repeats exactly this analysis.

In [ ]:
ANALYSIS, DETECTIONS = universal_pipeline.analyse_scene(source_image.linear, CONFIG, PROJECT)
CLASSIFICATION = ANALYSIS["classification"]
display(Markdown(report.analysis_markdown(ANALYSIS, universal_pipeline.preset_record(CONFIG, CLASSIFICATION))))
scene_path = universal_pipeline.scene_figure(source_image.linear, DETECTIONS, STAGING / "scene.png")
display(Image(filename=str(scene_path)))
if CLASSIFICATION["needs_confirmation"]:
    print(
        f"CONFIRM: low confidence, so the conservative general settings will be used. The automatic decision was "
        f"{CLASSIFICATION['detected']!r}; set CATEGORY in section 2 to confirm or correct it."
    )
del source_image  # the pipeline ingests again from the read-only copy

## 9. Baseline reconstruction

Runs the Phase 4 universal pipeline, which covers sections 9 to 14 in one call: ingest, scene analysis,
the selected route (the notebook 01 or 02 pipeline, or the common pipeline: Lanczos baseline, SR ladder,
combined masks, universal processing), the masters, the standardised evaluation and the final image. Every
stage is logged with its PRD 5.2 classification, duration and peak memory; a delegated specialised run logs its
own stages in its own manifest and report. If a stage fails, outputs already written are kept and the report
names the failing stage (PRD 22.3).

In [ ]:
try:
    RESULT = universal_pipeline.run_universal(SOURCE, CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION)
except pipeline.StageError as error:
    print(f"FAILED in stage {error.stage}: {error}")
    if error.manifest_path:
        print(f"partial manifest and report: {error.manifest_path.parent}")
    print(
        "Reduced-memory retry: set RUNTIME_PRESET='quick' in section 2, or a smaller super_resolution tile_size in "
        "ADVANCED (for example {'super_resolution': {'tile_size': 256}}), then re-run section 2 and every section "
        "after it. Outputs already written are kept."
    )
    raise
MANIFEST = RESULT.manifest
ROUTE = MANIFEST["route"]
DELEGATED = json.loads(Path(ROUTE["delegate"]["manifest"]).read_text()) if ROUTE.get("delegate") else None
print(
    f"run {RESULT.run_id}: {MANIFEST['status']} in {MANIFEST['total_seconds']:.1f} s, "
    f"peak RSS {MANIFEST['memory']['pipeline_peak_rss_mb']:.0f} MB; route: {ROUTE['pipeline']}"
)
display(Markdown(report.stage_section(MANIFEST)))

## 10. Pretrained SR

Inference only (PRD 10), through the Phase 2 backend shared with notebooks 00 to 02. In the common pipeline each
ladder rung is measured against the Lanczos baseline and kept only when it passes the seam gate and the
measurable-improvement gates (detail gain, source consistency, clipping); the first rung that passes is the
processing input, otherwise Lanczos. A delegated human or night run whose SR input failed its seam gate is
re-run without SR. The tables show the candidates or attempts, then each SR mode's device, tiling and seam
gate, and the model records (`docs/models.md`).

In [ ]:
display(Markdown(report.reconstruction_section(MANIFEST)))
sr_record = (DELEGATED or MANIFEST).get("super_resolution") or {}
modes = {m: s for m, s in (sr_record.get("modes") or {}).items() if "tiling" in s or "seams" in s}
if not modes:
    if not CONFIG.sr_enabled:
        why = "ENABLE_SUPER_RESOLUTION is off"
    elif sr_record.get("skipped"):
        why = sr_record["skipped"]
    elif sr_record.get("backend") == "lanczos_stand_in":
        why = "the human pipeline was re-run with its Lanczos stand-in after a failed seam gate"
    else:
        why = "the delegated run's SR input failed its seam gate and it was re-run without SR"
    print(f"no super-resolution in the final route: {why}")
else:
    display(
        table(
            ["mode", "weights", "tiles", "inference s", "hard seam", "seam gate"],
            [
                (
                    m,
                    f"`{s.get('weights', '-')}`",
                    (s.get("tiling") or {}).get("tiles", "-"),
                    f"{(s.get('tiling') or {}).get('inference_seconds', 0):.0f}",
                    (s.get("seams") or {}).get("max_score", "-"),
                    "pass" if (s.get("seams") or {}).get("passed") else "FAIL",
                )
                for m, s in modes.items()
            ],
        )
    )
for mode in modes:
    for folder in (RESULT.reports_dir, Path(ROUTE["delegate"]["report"]).parent if DELEGATED else None):
        heatmap = folder / "figures" / f"sr_difference_{mode}.png" if folder else None
        if heatmap and heatmap.exists():
            print(f"|{mode} SR - Lanczos| (synthesised detail and drift)")
            display(Image(filename=str(heatmap)))

## 11. Specialized reconstruction

The category's processing. A human scene was processed by the notebook 01 pipeline (identity protection,
crowd consistency, region masks) and a night scene by the notebook 02 pipeline (Layer 5B night processing with
the failure rule); their own reports hold every detail and are linked above. The common pipeline shows its
combined masks (PRD 8.3 luminance regions plus feathered face masks), the face policy and the per-face
consistency diagnostics.

In [ ]:
if DELEGATED is None:
    masks = ROUTE.get("masks", {})
    regions = masks.get("regions", {}).get("fractions", {})
    display(table(["region", "share of source pixels"], [(name, f"{share:.2%}") for name, share in regions.items()]))
    print(f"face masks: {len(masks.get('faces', []))}" + (f". {masks['face_policy']}" if masks.get("faces") else ""))
    for key in ("regions", "fallbacks"):
        if MANIFEST["figures"].get(key):
            display(Image(filename=MANIFEST["figures"][key]))
else:
    print(f"processed by the {ROUTE['pipeline']}; full report: {ROUTE['delegate']['report']}")
    display(
        table(
            ["delegated stage", "class (PRD 5.2)", "seconds"],
            [(s["name"], s["classification"], f"{s['seconds']:.2f}") for s in DELEGATED.get("stages", [])],
        )
    )

## 12. Tonal and optical processing

The tone, colour and optical steps with the provenance label each output carries. The common pipeline uses the
Layer 5B engine with the universal policy (no shadow lift or colour change, a low highlight shoulder for
clipping protection, debanding, low region-aware sharpening) or, for a night scene with faces, the night
profile's policy outside the face masks. A night scene shows the notebook 02 steps; a human scene the notebook
01 tonal contrast, colour balance and grain records.

In [ ]:
if DELEGATED is None:
    steps = ROUTE["processing"]["stages"]
    information = ROUTE["processing"]["information"]
elif "night" in DELEGATED:
    steps, information = DELEGATED["night"]["stages"], DELEGATED["night"]["information"]
else:
    steps = [
        {"name": s["name"], "enabled": True, "provenance": s["classification"], "note": "; ".join(s["notes"])}
        for s in DELEGATED["stages"]
        if s["name"] in ("tonal_contrast", "colour_balance", "grain", "edge_sharpening", "skin_texture_smoothing")
    ]
    information = (DELEGATED.get("provenance") or {}).get("information", {})
display(
    table(
        ["step", "enabled", "provenance", "note"],
        [(s["name"], "yes" if s["enabled"] else "no", s["provenance"], s.get("note") or "-") for s in steps],
    )
)
display(table(["information", "steps"], list(information.items())))

## 13. Optional sensor simulation

Not part of Phase 4 (PRD Phase 5, notebook 05). No sensor noise or synthetic DNG is produced here.

## 14. Professional exports

16-bit sRGB TIFF (embedded sRGB ICC profile, lossless, provenance in ImageDescription) and OpenEXR (linear
Rec.709, half by default) masters, each re-read and checked after writing (PRD 17.1, 23.6), with JSON sidecars.
The final image is the route's master saved again under `final/` (a hard link where possible) and verified by
SHA-256.

In [ ]:
display(
    table(
        ["route master", "kind", "size", "checks"],
        [
            (
                f"`{Path(e['path']).name}`",
                e["kind"],
                f"{e['size_bytes'] / 1e6:.1f} MB",
                f"{sum(e['checks'].values())}/{len(e['checks'])}",
            )
            for e in ROUTE["exports"].values()
        ],
    )
)
display(Markdown(report.final_section(MANIFEST)))
print(f"final image folder: {RESULT.masters_dir / final_output.FINAL_DIR}")

## 15. Quality metrics

The standardised evaluation, the same for every route: the final 16-bit TIFF master against the Lanczos
baseline at the same size. Detail energy (variance of the Laplacian of sRGB luma), source consistency after
area reduction to the source size (PSNR, SSIM, CIEDE2000; PRD 23.2) and the share of clipped samples, with the
provisional measurable-improvement gates. The specialised reports add their own metrics (identity consistency,
night metrics).

In [ ]:
display(Markdown(report.evaluation_section(MANIFEST)))

## 16. Visual comparison

The final image, then identical crops at 100% output pixels: source (nearest-neighbour), Lanczos, each SR
candidate, the final image and the amplified difference between processing input and final, and for a delegated
run its own face or night comparisons. Review them at 100% and 200% for halos, ringing, seams, invented texture
and face changes (PRD 25.2, 26.2).

In [ ]:
for caption, path in report.review_figures(MANIFEST):
    print(caption)
    display(Image(filename=path))

## 17. Report generation

The Markdown report (PRD 27: final image, classification and why, route, evaluation, crops, stage log) and the
machine-readable manifest are written next to their figures; a delegated run's own report is linked from it.

In [ ]:
print(f"report:   {RESULT.report_path}")
print(f"manifest: {RESULT.manifest_path}")
display(Markdown("\n".join(f"- {line}" for line in report.universal_conclusions(MANIFEST))))

## 18. Download and archive

**The final image.** The route's one final master, saved again under `final/` in this run's masters folder
with `final_` in front of its name by the shared final-output step (`final_output.publish`, the same step as
notebooks 00 to 02), verified byte for byte, and shown below.

**Delivery** (`final_output.deliver`). In Colab, `DOWNLOAD_FINAL` downloads the final 16-bit TIFF: **about
235 MB** for a standard 50 MP run, so the browser download takes a few minutes (if it stalls, take the file
from Drive or the Files panel). With Drive mounted (`MOUNT_DRIVE`), the final TIFF, EXR and their provenance
sidecars are copied to `FINAL_DRIVE_DIR/<run id>`, unless the masters are on Drive already.

**Report archive.** The reports (this run's and any delegated run's) are zipped and downloaded separately
(PRD 21.5); the zip holds no masters.

In [ ]:
FINAL = MANIFEST.get("final_output") or {}
if FINAL.get("files"):
    display(Markdown(f"**Final 50 MP image:** {FINAL['reason']}."))
    if MANIFEST["figures"].get("final"):
        display(Image(filename=MANIFEST["figures"]["final"]))
    for f in FINAL["files"].values():
        print(f"{Path(f['path']).name}: {f['width']} x {f['height']}, {f['size_bytes'] / 1e6:,.0f} MB ({f['path']})")
DELIVERY = final_output.deliver(MANIFEST, download=DOWNLOAD_FINAL, drive_dir=FINAL_DRIVE_DIR if MOUNT_DRIVE else None)
archive = universal_pipeline.archive_reports(MANIFEST, RESULT.reports_dir)
print(f"report archive: {archive}")
if IN_COLAB:
    from google.colab import files

    files.download(str(archive))